# FakeCLR Metrics — Fast Version
Calculates FID and LPIPS directly inside the notebook. No external script calls.
Expect ~2-3 minutes per snapshot total.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lpips', 'scipy', 'matplotlib', 'numpy', '--quiet'])
print('Packages ready!')

Packages ready!


In [2]:
import os

# Forcing Python to find the Microsoft C++ compiler
msvc_path = r"C:\Program Files\Microsoft Visual Studio\18\Insiders\VC\Tools\MSVC\14.50.35717\bin\Hostx64\x64\cl.exe"
os.environ['PATH'] += f";{msvc_path}"

In [3]:
import os, glob, re, pickle, io, zipfile
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import lpips
from scipy import linalg

import sys
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))


FAKECLR_ROOT = os.path.join(BASE_DIR, "src", "FakeCLR")
ZIP_PATH     = os.path.join(FAKECLR_ROOT, 'data', 'panda.zip')
RUN_37 = os.path.join(FAKECLR_ROOT, 'results', '00037-panda-mirror-paper256-kimg200-batch8-resumecustom-freezed2')
RUN_38 = os.path.join(FAKECLR_ROOT, 'results', '00038-panda-mirror-paper256-kimg200-batch8-resumecustom-freezed4')
N_REAL = 100
N_FAKE = 200
TRUNC  = 0.7
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Run 37 exists:', os.path.exists(RUN_37))
print('Run 38 exists:', os.path.exists(RUN_38))

Device: cuda
Run 37 exists: True
Run 38 exists: True


In [4]:
def get_snapshots(run_dir):
    snaps = []
    for s in glob.glob(os.path.join(run_dir, 'network-snapshot-*.pkl')):
        m = re.search(r'network-snapshot-(\d+)\.pkl', s)
        if m:
            tick = int(m.group(1))
            snaps.append({'path': s, 'tick': tick, 'kimg': tick * 4})
    return sorted(snaps, key=lambda x: x['kimg'])

snaps_37 = get_snapshots(RUN_37)
snaps_38 = get_snapshots(RUN_38)
last_kimg_37 = snaps_37[-1]['kimg'] if snaps_37 else 0

all_snapshots = snaps_37.copy()
for s in snaps_38:
    all_snapshots.append({'path': s['path'], 'tick': s['tick'], 'kimg': last_kimg_37 + s['kimg'], 'run': '38'})
for s in all_snapshots:
    if 'run' not in s:
        s['run'] = '37'

print(f'Run 37: {len(snaps_37)} snapshots')
print(f'Run 38: {len(snaps_38)} snapshots')
print(f'Total:  {len(all_snapshots)} snapshots, kimg {all_snapshots[0]["kimg"]} to {all_snapshots[-1]["kimg"]}')

Run 37: 13 snapshots
Run 38: 15 snapshots
Total:  28 snapshots, kimg 0 to 416


In [5]:
def load_real_images(zip_path, n=100, size=256):
    images = []
    transform = T.Compose([T.Resize((size, size)), T.ToTensor()])
    with zipfile.ZipFile(zip_path, 'r') as z:
        img_files = [f for f in z.namelist() if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for f in img_files[:n]:
            with z.open(f) as img_file:
                img = Image.open(io.BytesIO(img_file.read())).convert('RGB')
                images.append(transform(img))
    return torch.stack(images)

real_imgs_01 = load_real_images(ZIP_PATH, n=N_REAL)
real_imgs_lpips = (real_imgs_01 * 2 - 1).to(DEVICE)
print(f'Loaded {len(real_imgs_01)} real images')

# Inception for FID
import torch.nn.functional as F
from torchvision.models import inception_v3
inception = inception_v3(pretrained=True, transform_input=False).to(DEVICE).eval()
inception.fc = torch.nn.Identity()

def get_feats(imgs_01):
    feats = []
    imgs = F.interpolate(imgs_01, size=(299,299), mode='bilinear', align_corners=False)
    with torch.no_grad():
        for i in range(0, len(imgs), 16):
            feats.append(inception(imgs[i:i+16].to(DEVICE)).cpu().numpy())
    return np.concatenate(feats)

def fid_score(real_f, fake_f):
    mu_r, sig_r = real_f.mean(0), np.cov(real_f, rowvar=False)
    mu_f, sig_f = fake_f.mean(0), np.cov(fake_f, rowvar=False)
    diff = mu_r - mu_f
    cov, _ = linalg.sqrtm(sig_r @ sig_f, disp=False)
    if np.iscomplexobj(cov): cov = cov.real
    return float(diff @ diff + np.trace(sig_r + sig_f - 2*cov))

loss_lpips = lpips.LPIPS(net='alex').to(DEVICE)

print('Computing real image features...')
real_feats = get_feats(real_imgs_01)
print('Setup complete!')

Loaded 100 real images
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\vishw\OneDrive\Desktop\Explo\FakeCLR\fakeclr_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\vishw\OneDrive\Desktop\Explo\FakeCLR\fakeclr_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Users\vishw\OneDrive\Desktop\Explo\FakeCLR\fakeclr_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=

Loading model from: c:\Users\vishw\OneDrive\Desktop\Explo\FakeCLR\fakeclr_env\lib\site-packages\lpips\weights\v0.1\alex.pth
Computing real image features...


c:\Users\vishw\OneDrive\Desktop\Explo\FakeCLR\fakeclr_env\lib\site-packages\lpips\lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load

Setup complete!


In [ ]:
def generate_fakes(pkl_path, n=200, trunc=0.7):
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    G = data['G_ema'].to(DEVICE).eval()
    imgs = []
    with torch.no_grad():
        for i in range(0, n, 8):
            b = min(8, n-i)
            z = torch.randn(b, G.z_dim, device=DEVICE)
            c = torch.zeros(b, G.c_dim, device=DEVICE) if G.c_dim > 0 else None
            out = G(z, c, truncation_psi=trunc)
            imgs.append(((out.clamp(-1,1)+1)/2).cpu())
    del G; torch.cuda.empty_cache()
    return torch.cat(imgs)

results = []
for i, snap in enumerate(all_snapshots):
    kimg, run = snap['kimg'], snap['run']
    print(f'[{i+1}/{len(all_snapshots)}] kimg={kimg} run={run}... ', end='', flush=True)
    try:
        fakes = generate_fakes(snap['path'], n=N_FAKE, trunc=TRUNC)
        fid = fid_score(real_feats, get_feats(fakes))
        fakes_lp = (fakes[:N_REAL]*2-1).to(DEVICE)
        lp = float(np.mean([loss_lpips(real_imgs_lpips[j:j+1], fakes_lp[j:j+1]).item() for j in range(min(N_REAL, len(fakes_lp)))]))
        results.append({'kimg': kimg, 'run': run, 'fid': fid, 'lpips': lp})
        print(f'FID={fid:.1f}  LPIPS={lp:.4f}')
    except Exception as e:
        print(f'ERROR: {e}')

print('\nAll done!')

[1/28] kimg=0 run=37... Setting up PyTorch plugin "bias_act_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin

In [ ]:
r37 = [r for r in results if r['run']=='37']
r38 = [r for r in results if r['run']=='38']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('FakeCLR Metrics — Panda 100\nBlue=Run37(freezed=2)  Red=Run38(freezed=4)', fontsize=14, fontweight='bold')

for ax, metric, title in zip(axes, ['fid','lpips'], ['FID ↓','LPIPS ↓']):
    if r37: ax.plot([r['kimg'] for r in r37], [r[metric] for r in r37], 'b-o', label='Run37 freezed=2', markersize=6, lw=2)
    if r38: ax.plot([r['kimg'] for r in r38], [r[metric] for r in r38], 'r-o', label='Run38 freezed=4', markersize=6, lw=2)
    ax.axvline(x=last_kimg_37, color='gray', ls='--', alpha=0.5, label='Transition')
    ax.set_xlabel('kimg', fontsize=12); ax.set_ylabel(metric.upper(), fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fakeclr_metrics_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fakeclr_metrics_plot.png')

In [ ]:
print(f'{"kimg":>8} {"Run":>6} {"FID":>10} {"LPIPS":>10}')
print('-'*40)
for r in results:
    print(f'{r["kimg"]:>8} {r["run"]:>6} {r["fid"]:>10.2f} {r["lpips"]:>10.4f}')
if results:
    bf = min(results, key=lambda x: x['fid'])
    bl = min(results, key=lambda x: x['lpips'])
    print(f'\nBest FID:   {bf["fid"]:.2f} at kimg={bf["kimg"]} (Run {bf["run"]})')
    print(f'Best LPIPS: {bl["lpips"]:.4f} at kimg={bl["kimg"]} (Run {bl["run"]})')